# GPT-Small for CNN/DailyMail Summarization

A complete decoder-only GPT training pipeline, designed to run on an Apple Silicon Mac or Google Colab.

**Recommended first run:** use the smoke-test settings (1,000 train / 100 validation / 50 test examples, 1 epoch). After confirming that the pipeline works, restore the full settings.

**M1 note:** start with `BATCH_SIZE = 2` and `MAX_LENGTH = 512`. Training 50K examples for multiple epochs can take a long time, especially on an 8GB M1.


In [ ]:
!nvidia-smi

In [ ]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("Built with CUDA:", tf.test.is_built_with_cuda())
print("Physical GPUs:", tf.config.list_physical_devices("GPU"))

print("GPU device details:")
for gpu in tf.config.list_physical_devices("GPU"):
    print(gpu)

In [ ]:
"""
GPT-Small CNN/DailyMail summarization from scratch.
Apple Silicon install:
python3 -m pip install tensorflow tensorflow-metal datasets sentencepiece rouge-score tqdm

Colab install:
pip install tensorflow datasets sentencepiece rouge-score tqdm

Run: python gpt_small_summarization.py

M1-friendly starting settings: 10k vocab, 512 tokens, batch 2.
Training 50k examples for 10 epochs may take a long time on an M1.
"""

import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import json, math, random, time
from pathlib import Path

import numpy as np
import tensorflow as tf
import sentencepiece as spm
from datasets import load_dataset
from tqdm.auto import tqdm
from rouge_score import rouge_scorer

# -------------------- CONFIG --------------------
SEED = 42
PROJECT = Path("./GPT_Small_Experiment_2")
TOKENIZER_DIR = PROJECT / "tokenizer"
CHECKPOINT_DIR = PROJECT / "checkpoints"
OUTPUT_DIR = PROJECT / "outputs"
for p in (PROJECT, TOKENIZER_DIR, CHECKPOINT_DIR, OUTPUT_DIR):
    p.mkdir(parents=True, exist_ok=True)

DATASET = "abisee/cnn_dailymail"
CONFIG = "3.0.0"
TRAIN_SIZE, VAL_SIZE, TEST_SIZE =30_000, 2_000, 1_000

VOCAB_SIZE = 10_000
MAX_LENGTH = 512
MAX_SUMMARY_TOKENS = 96
D_MODEL, NUM_HEADS, NUM_LAYERS, D_FF = 256, 8, 6, 1024
DROPOUT = 0.1

BATCH_SIZE = 32         # M1 starting point; try 4 only if memory allows
EPOCHS = 10
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 500
CLIP_NORM = 1.0
PATIENCE = 3

# Smoke test: temporarily use TRAIN_SIZE=1000, VAL_SIZE=100,
# TEST_SIZE=50, EPOCHS=1 before launching the full run.

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## CONFIG


In [ ]:
# -------------------- CONFIG --------------------
SEED = 42
PROJECT = Path("./GPT_Small_Experiment_2")
TOKENIZER_DIR = PROJECT / "tokenizer"
CHECKPOINT_DIR = PROJECT / "checkpoints"
OUTPUT_DIR = PROJECT / "outputs"
for p in (PROJECT, TOKENIZER_DIR, CHECKPOINT_DIR, OUTPUT_DIR):
    p.mkdir(parents=True, exist_ok=True)

DATASET = "abisee/cnn_dailymail"
CONFIG = "3.0.0"
TRAIN_SIZE, VAL_SIZE, TEST_SIZE = 20_000, 1_000, 5_000

VOCAB_SIZE = 10_000
MAX_LENGTH = 512
MAX_SUMMARY_TOKENS = 96
D_MODEL, NUM_HEADS, NUM_LAYERS, D_FF = 256, 8, 6, 1024
DROPOUT = 0.1

BATCH_SIZE = 32          # M1 starting point; try 4 only if memory allows
EPOCHS = 10
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 500
CLIP_NORM = 1.0
PATIENCE = 3

# Smoke test: temporarily use TRAIN_SIZE=1000, VAL_SIZE=100,
# TEST_SIZE=50, EPOCHS=1 before launching the full run.

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## DEVICE


In [ ]:
# -------------------- DEVICE --------------------
print("TensorFlow:", tf.__version__)
print("Devices:", tf.config.list_physical_devices())
if not tf.config.list_physical_devices("GPU"):
    print("WARNING: TensorFlow GPU not detected; CPU training will be slow.")
else:
    print("TensorFlow GPU detected.")

## DATA


In [ ]:
# -------------------- DATA --------------------
print("Loading CNN/DailyMail...")
train_full = load_dataset(DATASET, CONFIG, split="train")
val_full = load_dataset(DATASET, CONFIG, split="validation")
test_full = load_dataset(DATASET, CONFIG, split="test")

train_data = train_full.shuffle(seed=SEED).select(range(TRAIN_SIZE))
val_data = val_full.shuffle(seed=SEED).select(range(VAL_SIZE))
test_data = test_full.shuffle(seed=SEED).select(range(TEST_SIZE))
print("Subset sizes:", len(train_data), len(val_data), len(test_data))

## TOKENIZER


In [ ]:
# -------------------- TOKENIZER --------------------
prefix = str(TOKENIZER_DIR / "gpt_tokenizer")
model_file = prefix + ".model"
corpus_file = PROJECT / "tokenizer_corpus.txt"

if not os.path.exists(model_file):
    with open(corpus_file, "w", encoding="utf-8") as f:
        for row in tqdm(train_data, desc="Building tokenizer corpus"):
            article = " ".join(row["article"].split())
            summary = " ".join(row["highlights"].split())
            if article:
                f.write(article + "\n")
            if summary:
                f.write(summary + "\n")

    spm.SentencePieceTrainer.train(
        input=str(corpus_file),
        model_prefix=prefix,
        vocab_size=VOCAB_SIZE,
        model_type="bpe",
        character_coverage=1.0,
        pad_id=0, unk_id=1, bos_id=2, eos_id=3,
        pad_piece="<pad>", unk_piece="<unk>",
        bos_piece="<s>", eos_piece="</s>",
        input_sentence_size=1_000_000,
        shuffle_input_sentence=True,
        num_threads=max(2, os.cpu_count() or 4),
    )
else:
    print("Reusing existing tokenizer.")

tokenizer = spm.SentencePieceProcessor()
assert tokenizer.load(model_file)
assert tokenizer.get_piece_size() == VOCAB_SIZE
PAD_ID, UNK_ID, BOS_ID, EOS_ID = (
    tokenizer.pad_id(), tokenizer.unk_id(), tokenizer.bos_id(), tokenizer.eos_id()
)
assert (PAD_ID, UNK_ID, BOS_ID, EOS_ID) == (0, 1, 2, 3)
PROMPT_IDS = tokenizer.encode("Summarize:", out_type=int)
INPUT_LENGTH = MAX_LENGTH - 1
print("Special IDs:", PAD_ID, UNK_ID, BOS_ID, EOS_ID)

In [ ]:

print("TensorFlow:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

!nvidia-smi

In [ ]:
import tensorflow as tf

with tf.device('/GPU:0'):
    a = tf.random.normal((3000, 3000))
    b = tf.random.normal((3000, 3000))
    c = tf.matmul(a, b)

print("Result:", c.shape)
print("Executed on:", c.device)

## PREPROCESS


In [ ]:
# -------------------- PREPROCESS --------------------
def preprocess(example):
    article_ids = tokenizer.encode(" ".join(example["article"].split()), out_type=int)
    summary_ids = tokenizer.encode(" ".join(example["highlights"].split()), out_type=int)
    summary_ids = summary_ids[:MAX_SUMMARY_TOKENS]

    article_limit = MAX_LENGTH - len(PROMPT_IDS) - len(summary_ids) - 3
    article_ids = article_ids[:max(0, article_limit)]

    sequence = [BOS_ID] + PROMPT_IDS + article_ids + [EOS_ID] + summary_ids + [EOS_ID]
    x = sequence[:-1]
    y = sequence[1:]

    # Mask next-token labels through the article EOS.
    prefix_len = 1 + len(PROMPT_IDS) + len(article_ids) + 1
    y[:prefix_len - 1] = [-100] * (prefix_len - 1)

    x, y = x[:INPUT_LENGTH], y[:INPUT_LENGTH]
    padding = INPUT_LENGTH - len(x)
    x += [PAD_ID] * padding
    y += [-100] * padding
    return np.asarray(x, np.int32), np.asarray(y, np.int32)

sx, sy = preprocess(train_data[0])
assert np.any(sy != -100)
print("Preprocessing verified:", sx.shape, sy.shape)
first_target = int(np.where(sy != -100)[0][0])
print("First supervised input/target IDs:", sx[first_target], sy[first_target])

## TF.DATA


In [ ]:
# -------------------- TF.DATA --------------------
def generator(dataset):
    for row in dataset:
        yield preprocess(row)

signature = (
    tf.TensorSpec((INPUT_LENGTH,), tf.int32),
    tf.TensorSpec((INPUT_LENGTH,), tf.int32),
)

def make_dataset(dataset, training=False):
    ds = tf.data.Dataset.from_generator(
        lambda: generator(dataset), output_signature=signature
    )
    if training:
        ds = ds.shuffle(min(len(dataset), 2000), seed=SEED,
                        reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_data, True)
val_ds = make_dataset(val_data)

## GPT BLOCK


In [ ]:
# -------------------- GPT BLOCK --------------------
class GPTBlock(tf.keras.layers.Layer):
    def __init__(self, d_model, heads, d_ff, dropout):
        super().__init__()
        self.norm1 = tf.keras.layers.LayerNormalization(epsilon=1e-5)
        self.attn = tf.keras.layers.MultiHeadAttention(
            num_heads=heads, key_dim=d_model // heads, dropout=dropout
        )
        self.drop1 = tf.keras.layers.Dropout(dropout)
        self.norm2 = tf.keras.layers.LayerNormalization(epsilon=1e-5)
        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(d_ff, activation=tf.keras.activations.gelu),
            tf.keras.layers.Dense(d_model),
        ])
        self.drop2 = tf.keras.layers.Dropout(dropout)

    def call(self, x, attention_mask, training=False):
        h = self.norm1(x)
        h = self.attn(h, h, attention_mask=attention_mask, training=training)
        x = x + self.drop1(h, training=training)
        h = self.ffn(self.norm2(x), training=training)
        return x + self.drop2(h, training=training)

## GPT MODEL


In [ ]:
# -------------------- GPT MODEL --------------------
class GPTSmall(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.token_emb = tf.keras.layers.Embedding(VOCAB_SIZE, D_MODEL)
        self.pos_emb = tf.keras.layers.Embedding(INPUT_LENGTH, D_MODEL)
        self.emb_drop = tf.keras.layers.Dropout(DROPOUT)
        self.blocks = [
            GPTBlock(D_MODEL, NUM_HEADS, D_FF, DROPOUT)
            for _ in range(NUM_LAYERS)
        ]
        self.final_norm = tf.keras.layers.LayerNormalization(epsilon=1e-5)
        self.lm_head = tf.keras.layers.Dense(VOCAB_SIZE)

    def call(self, input_ids, training=False):
        seq_len = tf.shape(input_ids)[1]
        positions = tf.range(seq_len)
        x = self.token_emb(input_ids) + self.pos_emb(positions)[None, :, :]
        x = self.emb_drop(x, training=training)

        # Combine causal mask with key-padding mask.
        causal = tf.linalg.band_part(
            tf.ones((seq_len, seq_len), dtype=tf.bool), -1, 0
        )
        valid_keys = tf.not_equal(input_ids, PAD_ID)[:, None, :]
        mask = tf.logical_and(causal[None, :, :], valid_keys)

        for block in self.blocks:
            x = block(x, attention_mask=mask, training=training)
        return self.lm_head(self.final_norm(x))

model = GPTSmall()
_ = model(tf.zeros((1, INPUT_LENGTH), dtype=tf.int32))
params = model.count_params()
print(f"Parameters: {params:,}")
print(f"FP32 weights alone: {params * 4 / (1024**2):.1f} MiB")

## LOSS / METRICS


In [ ]:
# -------------------- LOSS / METRICS --------------------
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True, reduction="none"
)

def masked_loss(labels, logits):
    valid = tf.not_equal(labels, -100)
    safe_labels = tf.where(valid, labels, tf.zeros_like(labels))
    losses = loss_fn(safe_labels, logits)
    weights = tf.cast(valid, losses.dtype)
    return tf.reduce_sum(losses * weights) / (tf.reduce_sum(weights) + 1e-8)

def masked_accuracy(labels, logits):
    valid = tf.not_equal(labels, -100)
    pred = tf.argmax(logits, axis=-1, output_type=tf.int32)
    correct = tf.logical_and(valid, tf.equal(labels, pred))
    return tf.reduce_sum(tf.cast(correct, tf.float32)) / (
        tf.reduce_sum(tf.cast(valid, tf.float32)) + 1e-8
    )

## LR SCHEDULE / OPTIMIZER


In [ ]:
# -------------------- LR SCHEDULE / OPTIMIZER --------------------
steps_per_epoch = math.ceil(TRAIN_SIZE / BATCH_SIZE)
total_steps = steps_per_epoch * EPOCHS

class WarmupCosine(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, peak_lr, warmup, total):
        super().__init__()
        self.peak_lr = peak_lr
        self.warmup = float(max(1, warmup))
        self.total = float(max(warmup + 1, total))

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warm = self.peak_lr * (step + 1.0) / self.warmup
        progress = tf.clip_by_value(
            (step - self.warmup) / (self.total - self.warmup), 0.0, 1.0
        )
        cosine = self.peak_lr * 0.5 * (1.0 + tf.cos(math.pi * progress))
        return tf.where(step < self.warmup, warm, cosine)

lr_schedule = WarmupCosine(LEARNING_RATE, WARMUP_STEPS, total_steps)
optimizer = tf.keras.optimizers.AdamW(
    learning_rate=lr_schedule,
    weight_decay=WEIGHT_DECAY,
    global_clipnorm=CLIP_NORM,
)

## TRAINING STEPS


In [ ]:
# -------------------- TRAINING STEPS --------------------
@tf.function
def train_step(x, y):
    with tf.GradientTape() as tape:
        logits = model(x, training=True)
        loss = masked_loss(y, logits)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss, masked_accuracy(y, logits)

@tf.function
def val_step(x, y):
    logits = model(x, training=False)
    return masked_loss(y, logits), masked_accuracy(y, logits)

In [ ]:
print(train_step)
print("Function type:", type(train_step))

In [ ]:
for x, y in train_ds.take(1):
    result = train_step(x, y)

    print("Result:", result)
    print("Result type:", type(result))

## TRAIN LOOP


In [ ]:
# -------------------- TRAIN LOOP --------------------
best_path = CHECKPOINT_DIR / "best_gpt_small.weights.h5"
history_path = OUTPUT_DIR / "training_history.json"
history = {"train_loss": [], "val_loss": [], "train_accuracy": [], "val_accuracy": []}
best_val = float("inf")
bad_epochs = 0

with open(OUTPUT_DIR / "config.json", "w") as f:
    json.dump({
        "train_size": TRAIN_SIZE, "val_size": VAL_SIZE, "test_size": TEST_SIZE,
        "vocab_size": VOCAB_SIZE, "max_length": MAX_LENGTH,
        "d_model": D_MODEL, "heads": NUM_HEADS, "layers": NUM_LAYERS,
        "d_ff": D_FF, "batch_size": BATCH_SIZE, "epochs": EPOCHS,
    }, f, indent=2)

for epoch in range(EPOCHS):
    start = time.time()
    tr_loss, tr_acc = tf.keras.metrics.Mean(), tf.keras.metrics.Mean()

    for x, y in tqdm(train_ds, desc=f"Epoch {epoch+1}/{EPOCHS} train"):
        loss, acc = train_step(x, y)
        tr_loss.update_state(loss)
        tr_acc.update_state(acc)

    va_loss, va_acc = tf.keras.metrics.Mean(), tf.keras.metrics.Mean()
    for x, y in tqdm(val_ds, desc=f"Epoch {epoch+1}/{EPOCHS} validation"):
        loss, acc = val_step(x, y)
        va_loss.update_state(loss)
        va_acc.update_state(acc)

    tl, ta = float(tr_loss.result()), float(tr_acc.result())
    vl, va = float(va_loss.result()), float(va_acc.result())
    history["train_loss"].append(tl)
    history["train_accuracy"].append(ta)
    history["val_loss"].append(vl)
    history["val_accuracy"].append(va)

    model.save_weights(CHECKPOINT_DIR / f"gpt_small_epoch_{epoch+1}.weights.h5")
    if vl < best_val:
        best_val = vl
        bad_epochs = 0
        model.save_weights(best_path)
        print("Saved new best checkpoint.")
    else:
        bad_epochs += 1

    with open(history_path, "w") as f:
        json.dump(history, f, indent=2)

    print(f"\nEpoch {epoch+1}: train loss={tl:.4f}, val loss={vl:.4f}")
    print(f"Train acc={ta:.4f}, val acc={va:.4f}, best val={best_val:.4f}")
    print(f"Elapsed: {(time.time()-start)/60:.1f} minutes")

    if bad_epochs >= PATIENCE:
        print("Early stopping.")
        break

model.load_weights(best_path)
print("Loaded best checkpoint:", best_path)

In [ ]:
# Save current best weights
model.load_weights(best_path)

# New training configuration
CONTINUE_EPOCHS = 5
NEW_LEARNING_RATE = 1e-4
NEW_WARMUP_STEPS = 100

CONTINUE_STEPS = math.ceil(len(train_data) / BATCH_SIZE)
TOTAL_CONTINUE_STEPS = CONTINUE_STEPS * CONTINUE_EPOCHS

print("Training steps per epoch:", CONTINUE_STEPS)
print("Total continuation steps:", TOTAL_CONTINUE_STEPS)
print("New peak learning rate:", NEW_LEARNING_RATE)

In [ ]:
 # -------------------- NEW LEARNING RATE SCHEDULE --------------------

lr_schedule = WarmupCosine(
    peak_lr=NEW_LEARNING_RATE,
    warmup=NEW_WARMUP_STEPS,
    total=TOTAL_CONTINUE_STEPS
)

# -------------------- NEW OPTIMIZER --------------------

optimizer = tf.keras.optimizers.AdamW(
    learning_rate=lr_schedule,
    weight_decay=WEIGHT_DECAY,
    global_clipnorm=CLIP_NORM
)

# -------------------- RETRACE TRAINING STEP --------------------

@tf.function
def train_step(x, y):
    with tf.GradientTape() as tape:
        logits = model(x, training=True)
        loss = masked_loss(y, logits)

    grads = tape.gradient(loss, model.trainable_variables)

    optimizer.apply_gradients(
        zip(grads, model.trainable_variables)
    )

    return loss, masked_accuracy(y, logits)

# -------------------- VERIFY --------------------

print("New optimizer initialized.")
print("Optimizer iterations:", optimizer.iterations.numpy())
print("Learning rate at step 0:", float(lr_schedule(0).numpy()))
print("Learning rate at step 100:", float(lr_schedule(100).numpy()))
print("Learning rate at final step:",
      float(lr_schedule(TOTAL_CONTINUE_STEPS - 1).numpy()))

In [ ]:
# Get one batch
x_batch, y_batch = next(iter(train_ds))

# Record loss before update
logits_before = model(x_batch, training=False)
loss_before = masked_loss(y_batch, logits_before)

# Perform one training update
loss, acc = train_step(x_batch, y_batch)

print("Training loss:", float(loss.numpy()))
print("Training accuracy:", float(acc.numpy()))
print("Optimizer iterations:", int(optimizer.iterations.numpy()))
print("Learning rate:", float(lr_schedule(optimizer.iterations - 1).numpy()))

In [ ]:
# -------------------- CONTINUATION TRAINING --------------------

CONTINUE_HISTORY = {
    "train_loss": [],
    "val_loss": [],
    "train_accuracy": [],
    "val_accuracy": []
}

CONTINUE_BEST_PATH = CHECKPOINT_DIR / "best_gpt_small_continued.weights.h5"
CONTINUE_HISTORY_PATH = OUTPUT_DIR / "continuation_history.json"

best_val = float("inf")
bad_epochs = 0

for epoch in range(CONTINUE_EPOCHS):

    start = time.time()

    tr_loss = tf.keras.metrics.Mean()
    tr_acc = tf.keras.metrics.Mean()

    for x, y in tqdm(
        train_ds,
        desc=f"Continuation Epoch {epoch+1}/{CONTINUE_EPOCHS}"
    ):
        loss, acc = train_step(x, y)
        tr_loss.update_state(loss)
        tr_acc.update_state(acc)

    va_loss = tf.keras.metrics.Mean()
    va_acc = tf.keras.metrics.Mean()

    for x, y in tqdm(val_ds, desc="Validation"):
        loss, acc = val_step(x, y)
        va_loss.update_state(loss)
        va_acc.update_state(acc)

    tl = float(tr_loss.result())
    ta = float(tr_acc.result())
    vl = float(va_loss.result())
    va = float(va_acc.result())

    CONTINUE_HISTORY["train_loss"].append(tl)
    CONTINUE_HISTORY["train_accuracy"].append(ta)
    CONTINUE_HISTORY["val_loss"].append(vl)
    CONTINUE_HISTORY["val_accuracy"].append(va)

    model.save_weights(
        CHECKPOINT_DIR / f"continued_epoch_{epoch+1}.weights.h5"
    )

    if vl < best_val:
        best_val = vl
        bad_epochs = 0
        model.save_weights(CONTINUE_BEST_PATH)
        print("Saved new best continuation checkpoint.")
    else:
        bad_epochs += 1

    with open(CONTINUE_HISTORY_PATH, "w") as f:
        json.dump(CONTINUE_HISTORY, f, indent=2)

    print(f"\nEpoch {epoch+1}:")
    print(f"Train loss: {tl:.4f} | Validation loss: {vl:.4f}")
    print(f"Train accuracy: {ta:.4f} | Validation accuracy: {va:.4f}")
    print(f"Elapsed: {(time.time()-start)/60:.2f} minutes")

    if bad_epochs >= PATIENCE:
        print("Early stopping.")
        break

# Restore best continuation weights
model.load_weights(CONTINUE_BEST_PATH)

print("\nContinuation training completed.")
print("Best continuation validation loss:", best_val)

In [ ]:

import os
import json
import shutil
import zipfile
import tensorflow as tf
from pathlib import Path

# -------------------- BACKUP DIRECTORY --------------------

BACKUP_DIR = OUTPUT_DIR / "COMPLETE_GPT_BACKUP"
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

BACKUP_CHECKPOINTS = BACKUP_DIR / "checkpoints"
BACKUP_CHECKPOINTS.mkdir(exist_ok=True)

# -------------------- 1. SAVE BEST MODEL --------------------

# Save the best continuation weights separately.
model.load_weights(CONTINUE_BEST_PATH)

model.save_weights(
    BACKUP_DIR / "best_model.weights.h5"
)

# Also preserve the original experiment's best weights.
shutil.copy2(
    best_path,
    BACKUP_DIR / "original_best_model.weights.h5"
)

# -------------------- 2. SAVE TRAINING HISTORY --------------------

with open(BACKUP_DIR / "original_history.json", "w") as f:
    json.dump(history, f, indent=2)

with open(BACKUP_DIR / "continuation_history.json", "w") as f:
    json.dump(CONTINUE_HISTORY, f, indent=2)

# -------------------- 3. SAVE CONFIGURATION --------------------

config = {
    "train_size": TRAIN_SIZE,
    "val_size": VAL_SIZE,
    "test_size": TEST_SIZE,
    "vocab_size": VOCAB_SIZE,
    "max_length": MAX_LENGTH,
    "input_length": INPUT_LENGTH,
    "d_model": D_MODEL,
    "num_heads": NUM_HEADS,
    "num_layers": NUM_LAYERS,
    "d_ff": D_FF,
    "batch_size": BATCH_SIZE,
    "original_epochs": EPOCHS,
    "continuation_epochs": CONTINUE_EPOCHS,
    "new_learning_rate": NEW_LEARNING_RATE,
    "warmup_steps": NEW_WARMUP_STEPS,
    "total_continuation_steps": TOTAL_CONTINUE_STEPS,
    "weight_decay": WEIGHT_DECAY,
    "clip_norm": CLIP_NORM,
    "optimizer_iterations": int(optimizer.iterations.numpy()),
    "best_original_val_loss": min(history["val_loss"]),
    "best_continuation_val_loss": min(CONTINUE_HISTORY["val_loss"])
}

with open(BACKUP_DIR / "config.json", "w") as f:
    json.dump(config, f, indent=2)

# -------------------- 4. SAVE OPTIMIZER + MODEL STATE --------------------

# IMPORTANT:
# This saves the currently loaded best model weights and current optimizer
# state. See the note below about exact training resumption.

optimizer.build(model.trainable_variables)

checkpoint = tf.train.Checkpoint(
    model=model,
    optimizer=optimizer
)

manager = tf.train.CheckpointManager(
    checkpoint,
    str(BACKUP_CHECKPOINTS),
    max_to_keep=3
)

saved_checkpoint = manager.save(
    checkpoint_number=int(optimizer.iterations.numpy())
)

print("TensorFlow checkpoint:", saved_checkpoint)

# -------------------- 5. SAVE TOKENIZER --------------------

# Save the SentencePiece model if tokenizer is a SentencePieceProcessor.
if hasattr(tokenizer, "serialized_model_proto"):
    tokenizer_path = BACKUP_DIR / "sentencepiece.model"
    tokenizer_path.write_bytes(tokenizer.serialized_model_proto())
    print("Tokenizer saved:", tokenizer_path)

# -------------------- 6. COPY EXISTING CONFIG FILE --------------------

existing_config = OUTPUT_DIR / "config.json"

if existing_config.exists():
    shutil.copy2(
        existing_config,
        BACKUP_DIR / "original_config.json"
    )

# -------------------- 7. CREATE ZIP BACKUP --------------------

zip_path = OUTPUT_DIR / "GPT_COMPLETE_BACKUP.zip"

with zipfile.ZipFile(
    zip_path, "w", compression=zipfile.ZIP_DEFLATED
) as archive:

    for file in BACKUP_DIR.rglob("*"):
        if file.is_file():
            archive.write(
                file,
                file.relative_to(OUTPUT_DIR)
            )

print("\n" + "=" * 50)
print("COMPLETE BACKUP CREATED")
print("=" * 50)
print("Backup directory:", BACKUP_DIR)
print("ZIP file:", zip_path)
print("ZIP size (MB):", round(zip_path.stat().st_size / (1024**2), 2))


## GENERATION


In [ ]:
def generate_summary(article, max_new_tokens=100, no_repeat_ngram_size=3):

    article_ids = tokenizer.encode(
        " ".join(article.split()),
        out_type=int
    )

    # Reserve space for the maximum summary length
    article_limit = (
        MAX_LENGTH - len(PROMPT_IDS) - MAX_SUMMARY_TOKENS - 3
    )

    article_ids = article_ids[:article_limit]

    prefix_ids = [BOS_ID] + PROMPT_IDS + article_ids + [EOS_ID]

    generated = []

    for _ in range(max_new_tokens):

        context = (prefix_ids + generated)[-INPUT_LENGTH:]

        logits = model(
            tf.constant([context], dtype=tf.int32),
            training=False
        )

        next_logits = logits[0, -1, :].numpy()

        next_logits[PAD_ID] = -1e9
        next_logits[UNK_ID] = -1e9
        next_logits[BOS_ID] = -1e9

        n = no_repeat_ngram_size

        if n > 1 and len(generated) >= n - 1:
            current = tuple(generated[-(n-1):])

            for i in range(len(generated) - n + 1):
                if tuple(generated[i:i+n-1]) == current:
                    next_logits[generated[i+n-1]] = -1e9

        token_id = int(np.argmax(next_logits))

        if token_id == EOS_ID:
            break

        generated.append(token_id)

    return tokenizer.decode(generated)

## QUALITATIVE TEST


In [ ]:
# -------------------- QUALITATIVE TEST --------------------
sample = test_data[0]
prediction = generate_summary(sample["article"])
print("\nARTICLE:\n", sample["article"][:1200])
print("\nREFERENCE:\n", sample["highlights"])
print("\nGENERATED:\n", prediction)

## ROUGE EVALUATION


In [ ]:
# -------------------- ROUGE EVALUATION --------------------
scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"], use_stemmer=True
)
N_EVAL = min(TEST_SIZE, 100)  # Set to TEST_SIZE for full test-subset evaluation.
totals = {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0}

for i in tqdm(range(N_EVAL), desc="ROUGE"):
    row = test_data[i]
    pred = generate_summary(row["article"])
    scores = scorer.score(row["highlights"], pred)
    for key in totals:
        totals[key] += scores[key].fmeasure

results = {key: value / N_EVAL for key, value in totals.items()}
print("\nROUGE:", results)
with open(OUTPUT_DIR / "rouge_results.json", "w") as f:
    json.dump({"examples": N_EVAL, "scores": results}, f, indent=2)

print("\nFinished. Outputs are in:", PROJECT.resolve())

In [ ]:
model.save_weights(
    CHECKPOINT_DIR / f"gpt_small_epoch_{epoch+1}.weights.h5"
)

In [ ]:
model.save_weights(best_path)

In [ ]:
model.load_weights(best_path)

In [ ]:
print("Best checkpoint exists:", best_path.exists())
print("Checkpoint size (MB):", best_path.stat().st_size / (1024**2))

print("Current model weights loaded from best checkpoint?")
print("Optimizer iterations:", optimizer.iterations.numpy())

print("History epochs:", len(history["train_loss"]))
print("Best validation loss:", min(history["val_loss"]))

In [67]:
print("OUTPUT_DIR:", OUTPUT_DIR)
print("OUTPUT_DIR exists:", OUTPUT_DIR.exists())

print("\nFiles in OUTPUT_DIR:")
for f in OUTPUT_DIR.iterdir():
    print(f.name, "-", "Directory" if f.is_dir() else "File")

OUTPUT_DIR: GPT_Small_Experiment_2/outputs
OUTPUT_DIR exists: True

Files in OUTPUT_DIR:
COMPLETE_GPT_BACKUP - Directory
training_history.json - File
GPT_COMPLETE_BACKUP.zip - File
rouge_results.json - File
continuation_history.json - File
config.json - File


In [68]:
from IPython.display import FileLink, display

zip_path = OUTPUT_DIR / "GPT_COMPLETE_BACKUP.zip"

if zip_path.exists():
    print("Backup found:", zip_path)
    print("Size (MB):", round(zip_path.stat().st_size / (1024**2), 2))
    display(FileLink(str(zip_path)))
else:
    print("ZIP backup has not been created yet.")

Backup found: GPT_Small_Experiment_2/outputs/GPT_COMPLETE_BACKUP.zip
Size (MB): 175.82


/kaggle/working/GPT_Small_Experiment_2/outputs/GPT_COMPLETE_BACKUP.zip

In [70]:
import json
from pathlib import Path

TOKENIZER_DIR = BACKUP_DIR / "tokenizer"
TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)

# Save complete SentencePiece model
tokenizer_path = TOKENIZER_DIR / "sentencepiece.model"

tokenizer_path.write_bytes(
    tokenizer.serialized_model_proto()
)

# Save tokenizer configuration and special IDs
tokenizer_config = {
    "vocab_size": tokenizer.vocab_size(),
    "pad_id": PAD_ID,
    "unk_id": UNK_ID,
    "bos_id": BOS_ID,
    "eos_id": EOS_ID,
    "prompt_ids": PROMPT_IDS,
    "prompt_text": "Summarize:"
}

with open(TOKENIZER_DIR / "tokenizer_config.json", "w") as f:
    json.dump(tokenizer_config, f, indent=2)

print("Tokenizer saved:", tokenizer_path)
print("Vocabulary size:", tokenizer.vocab_size())
print("Special tokens and prompt configuration saved.")

Tokenizer saved: GPT_Small_Experiment_2/outputs/COMPLETE_GPT_BACKUP/tokenizer/sentencepiece.model
Vocabulary size: 10000
Special tokens and prompt configuration saved.


In [72]:

# -------------------- RESUMABLE CHECKPOINT SETUP --------------------

RESUME_DIR = OUTPUT_DIR / "RESUMABLE_TRAINING"
RESUME_DIR.mkdir(parents=True, exist_ok=True)

# Save model and optimizer together
training_checkpoint = tf.train.Checkpoint(
    model=model,
    optimizer=optimizer
)

checkpoint_manager = tf.train.CheckpointManager(
    training_checkpoint,
    str(RESUME_DIR / "checkpoints"),
    max_to_keep=5
)

print("Resumable checkpoint system initialized.")
print("Checkpoint directory:", RESUME_DIR)


Resumable checkpoint system initialized.
Checkpoint directory: GPT_Small_Experiment_2/outputs/RESUMABLE_TRAINING


In [73]:
with open(CONTINUE_HISTORY_PATH, "w") as f:
    json.dump(CONTINUE_HISTORY, f, indent=2)

In [74]:

# -------------------- SAVE COMPLETE TRAINING STATE --------------------

saved_path = checkpoint_manager.save(
    checkpoint_number=int(optimizer.iterations.numpy())
)

# Save training progress metadata
resume_metadata = {
    "completed_continuation_epochs": epoch + 1,
    "optimizer_iterations": int(optimizer.iterations.numpy()),
    "best_validation_loss": best_val,
    "bad_epochs": bad_epochs,
    "continuation_history": CONTINUE_HISTORY
}

with open(RESUME_DIR / "resume_metadata.json", "w") as f:
    json.dump(resume_metadata, f, indent=2)

print("Complete training state saved:", saved_path)


Complete training state saved: GPT_Small_Experiment_2/outputs/RESUMABLE_TRAINING/checkpoints/ckpt-3126


In [75]:
import os
import tensorflow as tf

checkpoint_dir = RESUME_DIR / "checkpoints"

print("Checkpoint files:")

for f in sorted(checkpoint_dir.iterdir()):
    print(f.name, f.stat().st_size, "bytes")

print("\nLatest checkpoint:")
print(tf.train.latest_checkpoint(str(checkpoint_dir)))

print("\nCheckpoint contents:")
for name, shape in tf.train.list_variables(
    tf.train.latest_checkpoint(str(checkpoint_dir))
):
    print(name, shape)

Checkpoint files:
checkpoint 172 bytes
ckpt-3126.data-00000-of-00001 120044107 bytes
ckpt-3126.index 18243 bytes

Latest checkpoint:
GPT_Small_Experiment_2/outputs/RESUMABLE_TRAINING/checkpoints/ckpt-3126

Checkpoint contents:
_CHECKPOINTABLE_OBJECT_GRAPH []
model/final_norm/beta/.ATTRIBUTES/VARIABLE_VALUE [256]
model/final_norm/gamma/.ATTRIBUTES/VARIABLE_VALUE [256]
model/lm_head/_kernel/.ATTRIBUTES/VARIABLE_VALUE [256, 10000]
model/lm_head/bias/.ATTRIBUTES/VARIABLE_VALUE [10000]
model/pos_emb/_embeddings/.ATTRIBUTES/VARIABLE_VALUE [511, 256]
model/token_emb/_embeddings/.ATTRIBUTES/VARIABLE_VALUE [10000, 256]
optimizer/_iterations/.ATTRIBUTES/VARIABLE_VALUE []
optimizer/_trainable_variables/10/.ATTRIBUTES/VARIABLE_VALUE [8, 32, 256]
optimizer/_trainable_variables/11/.ATTRIBUTES/VARIABLE_VALUE [256]
optimizer/_trainable_variables/12/.ATTRIBUTES/VARIABLE_VALUE [256]
optimizer/_trainable_variables/13/.ATTRIBUTES/VARIABLE_VALUE [256]
optimizer/_trainable_variables/14/.ATTRIBUTES/VARIABLE_

In [77]:

import os
import zipfile
from pathlib import Path
from IPython.display import display, FileLink

# Your complete experiment output directory
source_dir = Path(OUTPUT_DIR).resolve()

# Save ZIP outside OUTPUT_DIR to avoid including the ZIP in itself
zip_path = Path("/kaggle/working/GPT_Small_Experiment_2_LOCAL_BACKUP.zip")

assert source_dir.exists(), f"Output directory not found: {source_dir}"

# Create complete ZIP
with zipfile.ZipFile(
    zip_path, "w", compression=zipfile.ZIP_DEFLATED
) as archive:

    for file in source_dir.rglob("*"):
        if file.is_file():
            archive.write(
                file,
                arcname=Path("GPT_Small_Experiment_2") / file.relative_to(source_dir)
            )

print("Backup created successfully!")
print("Location:", zip_path)
print("Size (MB):", round(zip_path.stat().st_size / (1024**2), 2))

# Verify ZIP integrity
with zipfile.ZipFile(zip_path, "r") as archive:
    bad_file = archive.testzip()

assert bad_file is None, f"ZIP verification failed: {bad_file}"

print("ZIP integrity verified!")

# Generate clickable download link
display(FileLink(str(zip_path)))


Backup created successfully!
Location: /kaggle/working/GPT_Small_Experiment_2_LOCAL_BACKUP.zip
Size (MB): 456.78
ZIP integrity verified!


/kaggle/working/GPT_Small_Experiment_2_LOCAL_BACKUP.zip

In [78]:
from pathlib import Path
from IPython.display import FileLink, display

zip_path = Path("/kaggle/working/GPT_Small_Experiment_2_LOCAL_BACKUP.zip")

print("File exists:", zip_path.exists())

if zip_path.exists():
    print("File size (MB):", round(zip_path.stat().st_size / (1024**2), 2))
    print("Download link:")
    display(FileLink("GPT_Small_Experiment_2_LOCAL_BACKUP.zip"))
else:
    print("ZIP not found. Files currently in /kaggle/working:")
    for f in Path("/kaggle/working").iterdir():
        print(f.name)

File exists: True
File size (MB): 456.78
Download link:


/kaggle/working/GPT_Small_Experiment_2_LOCAL_BACKUP.zip

In [4]:
import os

print(os.listdir("/kaggle/input"))

[]
